In [22]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

In [2]:
titanic = sns.load_dataset("titanic")
titanic_df = titanic.copy()

In [3]:
titanic_df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [4]:
titanic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [5]:
# Duplicated values
titanic_df.duplicated().sum()

np.int64(107)

In [6]:
titanic_df = titanic_df.drop_duplicates()

In [7]:
# Split input & output and test & train
X = titanic_df[["pclass", "sex", "age", "fare", "embarked"]]
y = titanic_df["survived"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, shuffle=True, stratify=y)

In [9]:
# Filling missing values
numeric_imputer = SimpleImputer(strategy="median").set_output(transform="pandas")
X_train["age"] = numeric_imputer.fit_transform(X_train[["age"]])
X_test["age"] = numeric_imputer.transform(X_test[["age"]])

In [10]:
category_imputer = SimpleImputer(strategy="most_frequent").set_output(transform="pandas")
X_train["embarked"] = category_imputer.fit_transform(X_train[["embarked"]])
X_test["embarked"] = category_imputer.transform(X_test[["embarked"]])

In [11]:
# Feature engineering
# Encoding
# Label Encoding
le = LabelEncoder()
X_train["sex"] = le.fit_transform(X_train["sex"])
X_test["sex"] = le.transform(X_test["sex"])

In [12]:
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore").set_output(transform="pandas")
X_train_encoded = ohe.fit_transform(X_train[["embarked"]])
X_train = pd.concat([X_train.drop("embarked", axis=1), X_train_encoded], axis=1)
X_test_encoded = ohe.transform(X_test[["embarked"]])
X_test = pd.concat([X_test.drop("embarked", axis=1), X_test_encoded], axis=1)

In [13]:
# Model training - no pruning
model_dtc = DecisionTreeClassifier()
model_dtc.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [14]:
# Model evaluation
y_test_pred = model_dtc.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
cm = confusion_matrix(y_test, y_test_pred)

print("Decision Tree Classifier:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print(cm)

Decision Tree Classifier:
Accuracy: 0.7584745762711864
Precision: 0.7
Recall: 0.7216494845360825
F1: 0.7106598984771574
[[109  30]
 [ 27  70]]


In [15]:
# Model evaluation - training
y_train_pred = model_dtc.predict(X_train)

accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)
cm = confusion_matrix(y_train, y_train_pred)

print("Decision Tree Classifier:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print(cm)

# Model is overfit

Decision Tree Classifier:
Accuracy: 0.9817518248175182
Precision: 1.0
Recall: 0.9557522123893806
F1: 0.9773755656108597
[[322   0]
 [ 10 216]]


In [16]:
# Model training - Random forest 
rfc_model = RandomForestClassifier(
    oob_score=True,
    random_state=42,
)

rfc_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


In [17]:
# Model evaluation
y_test_pred = rfc_model.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
cm = confusion_matrix(y_test, y_test_pred)

print("Decision Tree Classifier:")
print("Accuracy:", accuracy)
print("Training accuracy:", rfc_model.oob_score_)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print(cm)

Decision Tree Classifier:
Accuracy: 0.7796610169491526
Training accuracy: 0.7645985401459854
Precision: 0.7227722772277227
Recall: 0.7525773195876289
F1: 0.7373737373737373
[[111  28]
 [ 24  73]]


In [18]:
# Model evaluation - training
y_train_pred = rfc_model.predict(X_train)

accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)
cm = confusion_matrix(y_train, y_train_pred)

print("Decision Tree Classifier:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print(cm)

Decision Tree Classifier:
Accuracy: 0.9817518248175182
Precision: 0.990909090909091
Recall: 0.9646017699115044
F1: 0.9775784753363229
[[320   2]
 [  8 218]]


In [23]:
# Model evaluation - pre pruning
rfc_cv_model = RandomizedSearchCV(
    RandomForestClassifier(
        oob_score=True,
        random_state=42,
        max_features=201,
    ),
    {
        "max_depth": np.arange(1, 20),
        "min_samples_split": np.arange(5, 50, 5),
    }
)
rfc_cv_model.fit(X_train, y_train)

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': array([ 1, 2... 18, 19]), 'min_samples_split': array([ 5, 10..., 35, 40, 45])}"
,n_iter,10
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [24]:
# Model evaluation - training
y_train_pred = rfc_cv_model.predict(X_train)

accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)
cm = confusion_matrix(y_train, y_train_pred)

print("Decision Tree Classifier:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print(cm)

Decision Tree Classifier:
Accuracy: 0.8211678832116789
Precision: 0.8478260869565217
Recall: 0.6902654867256637
F1: 0.7609756097560976
[[294  28]
 [ 70 156]]
